In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Button, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# LEAST-SQUARES IIR MODELING — THE CORE IDEA
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.ls-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.ls-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.ls-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.ls-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.ls-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
}

.ls-result{
    background:#eef7ee;
    border:1px solid #9cc79c;
}

.widget-label{
    font-size:13px !important;
}

.jupyter-widgets input{
    font-size:12.5px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ls-root">

<div class="ls-header">
Least-Squares IIR Modeling — The Core Idea
</div>

<div class="ls-doc">

The purpose of this notebook is to illustrate the basic modeling problem behind
least-squares IIR filter design.

The desired impulse response is

<div style="text-align:center;font-size:15.5px;margin:5px 0;">
<b>
h<sub>d</sub>[n] = 4(1/3)<sup>n</sup>u[n],
</b>
</div>

while the approximating first-order IIR system is

<div style="text-align:center;font-size:15.5px;margin:5px 0;">
<b>
H(z) = β₀ / (1 + α₁z<sup>-1</sup>),
&nbsp;&nbsp;
h[n] = β₀(-α₁)<sup>n</sup>u[n].
</b>
</div>

Move <b>β₀</b> and <b>α₁</b>. The left plot compares the desired and modeled impulse
responses, while the right plot shows the sample-by-sample approximation error

<b>e[n] = h<sub>d</sub>[n] - h[n]</b>.

The quantity displayed below the controls is the quadratic error

<div style="text-align:center;font-size:15.5px;margin:5px 0;">
<b>
E = Σ |e[n]|².
</b>
</div>

Least-squares design consists of selecting the model coefficients so that this
quantity becomes as small as possible. The exact model for this particular desired
sequence is obtained for <b>β₀ = 4</b> and <b>α₁ = -1/3</b>.

</div>

</div>
"""))

# ============================================================
# DESIRED IMPULSE RESPONSE
# ============================================================

L = 20

n = np.arange(L)

hd = 4.0*(1.0/3.0)**n

# ============================================================
# CONTROLS
# ============================================================

beta_slider = FloatSlider(value=3.0,min=0.0,max=6.0,step=0.02,description='β₀:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='280px'))

alpha_slider = FloatSlider(value=-0.10,min=-0.95,max=0.95,step=0.01,description='α₁:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='290px'))

exact_button = Button(description='Exact fit',button_style='',tooltip='Set β₀ = 4 and α₁ = -1/3',layout=Layout(width='105px'))

control_title = HTML('<div class="ls-title" style="margin:0;">Model parameters</div>',layout=Layout(width='125px'))

controls = HBox([control_title,beta_slider,alpha_slider,exact_button],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='8px 10px',margin='0 0 7px 0',align_items='center'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 7px 0'))

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(9.0,4.0))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# DESIRED AND MODELED IMPULSE RESPONSES
# ============================================================

marker_hd,stem_hd,base_hd = ax1.stem(n,hd,linefmt='C0-',markerfmt='C0o',basefmt=' ')

marker_h,stem_h,base_h = ax1.stem(n,np.zeros(L),linefmt='r-',markerfmt='ro',basefmt=' ')

plt.setp(stem_hd,linewidth=1.1)
plt.setp(stem_h,linewidth=1.1)

marker_hd.set_markersize(4.5)
marker_h.set_markersize(4.5)

marker_hd.set_label(r'Desired $h_d[n]$')
marker_h.set_label(r'Model $h[n]$')

ax1.axhline(0,color='black',linewidth=0.8)

ax1.set_xlim(-0.5,L-0.5)
ax1.set_ylim(-6.5,6.5)

ax1.set_title('Desired and Modeled Impulse Responses')
ax1.set_xlabel('Sample index n')
ax1.set_ylabel('Amplitude')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# APPROXIMATION ERROR
# ============================================================

marker_e,stem_e,base_e = ax2.stem(n,np.zeros(L),linefmt='r-',markerfmt='ro',basefmt=' ')

plt.setp(stem_e,linewidth=1.1)

marker_e.set_markersize(4.5)

ax2.axhline(0,color='black',linewidth=0.8)

ax2.set_xlim(-0.5,L-0.5)
ax2.set_ylim(-6.5,6.5)

ax2.set_title(r'Approximation Error $e[n]=h_d[n]-h[n]$')
ax2.set_xlabel('Sample index n')
ax2.set_ylabel('Error')

ax2.grid(True,linestyle=':',alpha=0.25)

plt.subplots_adjust(left=0.08,right=0.98,top=0.91,bottom=0.20,wspace=0.28)

# ============================================================
# UPDATE
# ============================================================

def update_model(change=None):

    beta0 = beta_slider.value
    alpha1 = alpha_slider.value

    h = beta0*(-alpha1)**n

    error = hd-h

    E = np.sum(np.abs(error)**2)

    pole = -alpha1

    marker_h.set_data(n,h)

    stem_h.set_segments([[[x,0],[x,y]] for x,y in zip(n,h)])

    marker_e.set_data(n,error)

    stem_e.set_segments([[[x,0],[x,y]] for x,y in zip(n,error)])

    info.value = f"""
    <div class="ls-root">

    <div class="ls-box ls-result">

    <b>Current model:</b>
    &nbsp;
    H(z) =
    <b>{beta0:.4f} / (1 {alpha1:+.4f}z<sup>-1</sup>)</b>

    &nbsp;&nbsp;&nbsp;

    Pole:
    <b>z = {pole:.4f}</b>

    &nbsp;&nbsp;&nbsp;

    Quadratic error:
    <b>E = {E:.8f}</b>

    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EXACT-FIT BUTTON
# ============================================================

def set_exact_fit(button):

    beta_slider.value = 4.0

    alpha_slider.value = -1.0/3.0

# ============================================================
# EVENTS
# ============================================================

beta_slider.observe(update_model,names='value')

alpha_slider.observe(update_model,names='value')

exact_button.on_click(set_exact_fit)

# ============================================================
# DISPLAY
# ============================================================

display(controls)

display(info)

display(fig.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_model()